<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets transformers

from datasets import load_dataset
ds = load_dataset("coastalcph/lex_glue", "unfair_tos")

for split in ["train", "validation", "test"]:
    texts = ds[split]["text"]
    empty = sum(1 for t in texts if not t.strip())
    dupes = len(texts) - len(set(texts))
    print(f"{split}: {empty} empty, {dupes} duplicates")

train: 0 empty, 105 duplicates
validation: 0 empty, 14 duplicates
test: 0 empty, 7 duplicates


In [ ]:
from collections import defaultdict, Counter

# 1. Most common duplicates in train
counts = Counter(ds["train"]["text"])
for text, n in counts.most_common(5):
    print(n, "x |", text[:80])

# 2. Same sentence, different labels?
labels_seen = defaultdict(set)
for t, l in zip(ds["train"]["text"], ds["train"]["labels"]):
    labels_seen[t].add(tuple(l))
conflicts = sum(1 for v in labels_seen.values() if len(v) > 1)
print("\nConflicting labels in train:", conflicts)

# 3. Test sentences that also appear in train
train_set = set(ds["train"]["text"])
overlap = sum(1 for t in ds["test"]["text"] if t in train_set)
print("Test sentences also in train:", overlap)

4 x | the copyright act provides that : 

3 x | please check the latest rates before you make your call . 

3 x | no refunds will be given for any charges more than 90 days old . 

3 x | you may opt out of this agreement to arbitrate . 

3 x | arbitration is more informal than a lawsuit in court . 


Conflicting labels in train: 1
Test sentences also in train: 33


As shown, there is some duplicates. But they are just common legal phrases that the companies reuse.

Also, Only 1 sentence in train has conflicting labels, which is too small to matter.

33 test sentences (about 2%) also appear in the training set. I kept the data unchanged so my results can be compared with the LexGLUE paper and I will check my results on the unseen test sentences later.

In [ ]:
names = ds["train"].features["labels"].feature.names

for i, name in enumerate(names):
    print(f"\n=== {name} ===")
    examples = [t for t, l in zip(ds["train"]["text"], ds["train"]["labels"]) if i in l]
    for t in examples[:2]:
        print("-", t[:200])


=== Limitation of liability ===
- though tinder strives to encourage a respectful user experience through features like the double opt-in that only allows users to communicate if they have both indicated interest in one another , it i
- although tinder reserves the right to review and remove content that violates this agreement , such content is the sole responsibility of the user who posts it , and tinder can not guarantee that all 

=== Unilateral termination ===
- this means we may add new product features or enhancements from time to time as well as remove some features , and if these actions do not materially affect your rights or obligations , we may not pro
- tinder may terminate your account at any time without notice if it believes that you have violated this agreement . 


=== Unilateral change ===
- we may make changes to this agreement and to the services from time to time . 

- the most recent version of this agreement will be posted on the services under settings and als

Some of the sentences can belong to more than one label. Also the unfair part is often later on in the sentences like the middle of it or the end, therefore the model will need to read the whole sentence and not just look for a keyword.

In [ ]:
from transformers import AutoTokenizer
import numpy as np

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
lengths = [len(tok(t)["input_ids"]) for t in ds["train"]["text"]]

print("Average:", round(np.mean(lengths)))
print("95% of sentences are under:", int(np.percentile(lengths, 95)))
print("Longest:", max(lengths))
for limit in [64, 128, 256]:
    over = sum(1 for n in lengths if n > limit)
    print(f"Longer than {limit}: {over}")

Average: 38
95% of sentences are under: 85
Longest: 477
Longer than 64: 611
Longer than 128: 70
Longer than 256: 5


The average sentence is 38 tokens and 95% are under 85 tokens. Only 70 sentences (about 1.3%) are longer than 128 tokens so I chose a maximum length of 128. This fits almost all sentences while keeping training fast.

In [ ]:
names = ds["train"].features["labels"].feature.names
num_labels = len(names)

def to_checklist(example):
    boxes = [0.0] * num_labels        # 8 empty boxes
    for label in example["labels"]:
        boxes[label] = 1.0            # tick each box that applies
    example["checklist"] = boxes
    return example

ds = ds.map(to_checklist)

# Check a few sentences: old label -> new checklist
for i in [0, 8, 10]:
    print(ds["train"][i]["labels"], "->", ds["train"][i]["checklist"])

# Check the total: should be 702 ticked boxes in train
total = sum(sum(row) for row in ds["train"]["checklist"])
print("Total ticked boxes in train:", int(total))

[] -> [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[4] -> [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[2] -> [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Total ticked boxes in train: 702


I converted each sentence's labels into a checklist of 8 boxes, one per unfair type where 1 means the type applies and 0 means it doesn't.
Fair sentences have all zeros. This lets the model answer 8 separate yes/no questions per sentence.
The total number of ticked boxes (702) matches the original label count so the conversion is correct.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tok(batch["text"], truncation=True, padding="max_length", max_length=128)

ds = ds.map(tokenize, batched=True)

# Look at one sentence after translation
ex = ds["train"][8]
print(ex["text"][:100])
print("\nTokens:", tok.convert_ids_to_tokens(ex["input_ids"])[:15])
print("IDs:", ex["input_ids"][:15])
print("Attention mask:", ex["attention_mask"][:15])
print("\nBox size:", len(ex["input_ids"]))
print("Real tokens:", sum(ex["attention_mask"]))

by creating a tinder account or by using the tinder imessage app ( `` tinder stacks '' ) , whether t

Tokens: ['[CLS]', 'by', 'creating', 'a', 'tin', '##der', 'account', 'or', 'by', 'using', 'the', 'tin', '##der', 'im', '##ess']
IDs: [101, 2011, 4526, 1037, 9543, 4063, 4070, 2030, 2011, 2478, 1996, 9543, 4063, 10047, 7971]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Box size: 128
Real tokens: 120


I tokenized the text with the bert-base-uncased tokenizer. Each sentence starts with a [CLS] token and is padded or cut to 128 tokens. Words that are not in BERT's dictionary, like 'tinder' are split into smaller pieces.
An attention mask tells the model which tokens are real and which are padding.

### **Data Augmentation**
**Protected word list** - A list of words that augmentation is never allowed to change, in order to keep an original legal meaning to a word and not replace it with something else similar but different.

In [ ]:
# Words the augmentation must never change, because swapping them
# would change the legal meaning (and make the label wrong)
PROTECTED = {
    # dispute and court words
    "arbitration", "arbitrator", "arbitral", "dispute", "disputes", "claim", "claims",
    "court", "courts", "jurisdiction", "venue", "jury", "judge", "lawsuit", "class", "action",
    # law words
    "law", "laws", "governed", "governing", "legal", "rights", "waive", "binding",
    # liability words
    "liability", "liable", "damages", "loss", "losses", "warranty", "indemnify", "negligence",
    # power and change words
    "terminate", "termination", "suspend", "remove", "removal", "modify", "change", "changes",
    "discretion", "sole", "solely", "notice", "consent", "agree", "agreement", "terms",
    # small words that flip or limit meaning
    "not", "no", "never", "only", "without", "any", "all", "always", "exclusive", "exclusively",
}

def protected_words_in(sentence):
    """Return the protected words found in a sentence."""
    return [w for w in sentence.split() if w in PROTECTED]

PROTECTED.update({"service", "services", "breach", "consumer", "association"})
PROTECTED.update({
    # places and courts (the core of Jurisdiction and Choice of law)
    "state", "states", "united", "federal", "county", "district", "california", "england",
    "wales", "canada", "new", "york", "san", "santa", "clara", "northern", "english",
    # fixed legal phrases
    "accordance", "regard", "conflict", "conflicts", "principles", "provision", "provisions",
    "parties", "party", "section", "personal", "individual", "subject", "small",
    "right", "actions", "extent", "relationship",
})
PROTECTED.update({
    "finland", "texas", "francisco", "mateo", "american", "country", "international",
    "resident", "residents", "user", "users", "company", "affiliates", "third", "choice",
    "applicable", "final", "controversy", "agreements", "privacy", "policy",
    "intellectual", "property", "trade", "publicity", "jams", "eula", "masquerade", "website",
})
PROTECTED.update({
    "irrevocably", "competent", "relief", "judicial", "mandatory", "member", "union",
    "european", "trademarks", "copyrights", "patents", "secrets", "corporate",
    "excluded", "finally", "act", "clause", "procedures",
})
PROTECTED.update({"well", "use", "language", "visit"})
PROTECTED.update({"representative", "waiver", "injunctive", "unenforceable", "objections",
                  "formation", "theory", "dutch", "sweden", "usa", "conditions"})
PROTECTED.update({"internal", "national", "reference"})

# Test on one real Arbitration sentence (label 7)
example = next(t for t, l in zip(ds["train"]["text"], ds["train"]["labels"]) if 7 in l)
print(example)
print("\nProtected words found:", protected_words_in(example))

the exclusive means of resolving any dispute or claim arising out of or relating to this agreement ( including any alleged breach thereof ) or the service shall be binding arbitration administered by the american arbitration association under the consumer arbitration rules . 


Protected words found: ['exclusive', 'any', 'dispute', 'claim', 'agreement', 'any', 'breach', 'service', 'binding', 'arbitration', 'american', 'arbitration', 'association', 'consumer', 'arbitration']


In [ ]:
from collections import Counter

rare_types = {5, 6, 7}   # Choice of law, Jurisdiction, Arbitration
swappable = Counter()

for text, labels in zip(ds["train"]["text"], ds["train"]["labels"]):
    if rare_types & set(labels):                      # sentence has a rare type
        for word, tag in nltk.pos_tag(text.split()):
            if wordnet_pos(tag) and word not in PROTECTED and word.isalpha():
                swappable[word] += 1                  # count words that could be swapped

print(swappable.most_common(40))

[('such', 21), ('other', 16), ('rules', 15), ('connection', 12), ('certain', 6), ('matter', 5), ('types', 5), ('purpose', 5), ('as', 4), ('way', 4), ('whatsapp', 4), ('skyscanner', 4), ('thereof', 3), ('otherwise', 3), ('card', 3), ('instagram', 3), ('effect', 3), ('also', 3), ('respect', 3), ('out', 3), ('niantic', 3), ('reason', 2), ('site', 2), ('credit', 2), ('general', 2), ('youtube', 2), ('tos', 2), ('mobile', 2), ('paragraph', 2), ('luxembourg', 2), ('exception', 2), ('dress', 2), ('domain', 2), ('names', 2), ('violations', 2), ('limitation', 2), ('expressly', 2), ('trial', 2), ('nations', 2), ('goods', 2)]


**Augmentation**

cell1. Load the tools and test swapping single words

cell2. Make new versions of one sentence, as a test. It takes a full sentence and makes a new version of it by swapping 1 or 2 safe words.

cell3. Run it on all the rare-type sentences

In [ ]:
!pip install -q gensim
import nltk, random
for pkg in ["wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
from nltk.corpus import wordnet as wn
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")   # word meanings, 100 numbers per word

def wordnet_pos(tag):
    """Turn NLTK's word-type tag into WordNet's. Verbs and others return None (never swapped)."""
    if tag.startswith("NN"): return wn.NOUN
    if tag.startswith("JJ"): return wn.ADJ
    if tag.startswith("RB") or tag == "RP": return wn.ADV
    return None

def best_synonym(word, tag):
    """Find the closest-meaning synonym for a word, or None if it shouldn't be swapped."""
    pos = wordnet_pos(tag)
    if pos is None or word in PROTECTED or word not in glove:
        return None
    if any(s.instance_hypernyms() for s in wn.synsets(word)):
        return None      # it's a name (place, organization), never swap
    base = wn.morphy(word, pos) or word          # e.g. "rules" -> "rule"
    candidates = []
    for syn in wn.synsets(word, pos=pos):
        for lemma in syn.lemma_names():
            lemma = lemma.lower()
            if ("_" in lemma or lemma in (word, base) or lemma in candidates or word.startswith(lemma) or lemma.startswith(word)):
                continue                          # skip multi-word, same word, repeats
            if lemma in glove:
                candidates.append(lemma)
    candidates = candidates[:10]                  # up to 10, like Perçin
    if not candidates:
        return None
    return max(candidates, key=lambda c: glove.similarity(word, c))

# Test on single words
for w, t in [("rules", "NNS"), ("alleged", "JJ"), ("means", "NNS"), ("arbitration", "NN")]:
    print(w, "->", best_synonym(w, t))

rules -> regulation
alleged -> supposed
means -> way
arbitration -> None


In [ ]:
random.seed(42)   # makes the random choices repeatable, so you get the same results every run

def augment_sentence(sentence, max_swaps=2):
    """Make one new version of a sentence by swapping 1 or 2 safe words."""
    words = sentence.split()
    tags = nltk.pos_tag(words)
    options = []                                   # (position, replacement) for every swappable word
    for i, (w, t) in enumerate(tags):
        syn = best_synonym(w, t)
        if syn:
            options.append((i, syn))
    if not options:
        return None                                # nothing safe to change
    n = random.randint(1, min(max_swaps, len(options)))
    for i, syn in random.sample(options, n):
        words[i] = syn
    return " ".join(words)

# Test: 3 new versions of the Arbitration example
print("ORIGINAL:", example, "\n")
for _ in range(3):
    new = augment_sentence(example)
    changes = [(a, b) for a, b in zip(example.split(), new.split()) if a != b]
    print("NEW:", new)
    print("Changed:", changes, "\n")

ORIGINAL: the exclusive means of resolving any dispute or claim arising out of or relating to this agreement ( including any alleged breach thereof ) or the service shall be binding arbitration administered by the american arbitration association under the consumer arbitration rules . 
 

NEW: the exclusive way of resolving any dispute or claim arising out of or relating to this agreement ( including any alleged breach thereof ) or the service shall be binding arbitration administered by the american arbitration association under the consumer arbitration rules .
Changed: [('means', 'way')] 

NEW: the exclusive way of resolving any dispute or claim arising out of or relating to this agreement ( including any alleged breach thereof ) or the service shall be binding arbitration administered by the american arbitration association under the consumer arbitration regulation .
Changed: [('means', 'way'), ('rules', 'regulation')] 

NEW: the exclusive means of resolving any dispute or claim ari

### **Run it on all rare-type sentences**

**What it does:**

Goes through every training sentence that has Choice of law, Jurisdiction or Arbitration

Tries to make 2 different new versions of each (up to 5 attempts)
Throws away any version that’s identical to the original, already in the dataset, or a repeat

Gives each new version the same labels as its original. If the original also had another type, like Jurisdiction and Choice of law, the new version keeps both

Prints a summary: how many examples each rare type had before, and how many after

In [ ]:
random.seed(42)   # same random choices every run
aug_texts, aug_labels, aug_sources = [], [], []

aug_texts, aug_labels = [], []
existing = set(ds["train"]["text"])
originals, skipped = 0, 0

for text, labels in zip(ds["train"]["text"], ds["train"]["labels"]):
    if rare_types & set(labels):                     # only Choice of law, Jurisdiction, Arbitration
        originals += 1
        made = set()
        for _ in range(5):                            # up to 5 tries to get 2 different versions
            new = augment_sentence(text)
            if new and new != text and new not in existing and new not in made:
                made.add(new)
            if len(made) == 2:
                break
        if not made:
            skipped += 1                              # nothing safe to change in this sentence
        for new in made:
            aug_texts.append(new)
            aug_labels.append(labels)                 # same labels as the original
            aug_sources.append(text)                  # remember the original sentence

print("Rare-type sentences:", originals)
print("New sentences made:", len(aug_texts))
print("Sentences that couldn't be changed:", skipped, "\n")

for i in sorted(rare_types):
    before = sum(1 for l in ds["train"]["labels"] if i in l)
    added = sum(1 for l in aug_labels if i in l)
    print(f"{names[i]}: {before} -> {before + added}")

Rare-type sentences: 95
New sentences made: 124
Sentences that couldn't be changed: 24 

Choice of law: 39 -> 81
Jurisdiction: 34 -> 76
Arbitration: 28 -> 75


## **Quality check**

**Automatic meaning check** - Legal-BERT turns each sentence into a list of numbers that represents its meaning. Then it measures how close the new sentence’s meaning is to the original, a score called cosine.

**similarity:**

1.0 = identical meaning, Lower = the meaning drifted

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

lb_tok = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
lb_model = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").eval()

def embed(sentences):
    """Turn sentences into meaning vectors (average of Legal-BERT's word vectors)."""
    enc = lb_tok(sentences, padding=True, truncation=True, max_length=128, return_tensors="pt")
    with torch.no_grad():
        out = lb_model(**enc).last_hidden_state
    mask = enc["attention_mask"].unsqueeze(-1)            # ignore padding
    return (out * mask).sum(1) / mask.sum(1)

sims = torch.nn.functional.cosine_similarity(embed(aug_sources), embed(aug_texts)).tolist()

print("Average similarity:", round(sum(sims) / len(sims), 3))
print("Lowest similarity:", round(min(sims), 3), "\n")

# The 5 new sentences that drifted the most
for i in sorted(range(len(sims)), key=lambda i: sims[i])[:5]:
    changes = [(a, b) for a, b in zip(aug_sources[i].split(), aug_texts[i].split()) if a != b]
    print(round(sims[i], 3), changes)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Average similarity: 0.996
Lowest similarity: 0.964 

0.964 [('also', 'likewise'), ('carefully', 'cautiously')]
0.971 [('also', 'likewise'), ('completely', 'totally')]
0.986 [('forms', 'variety'), ('part', 'portion')]
0.988 [('general', 'universal'), ('rules', 'regulation')]
0.989 [('site', 'website')]


**Hand check** - picks 20 random new sentences and shows only the part that changed, with a few words of context on each side, so I can judge each swap in a few seconds.

How to score each one (the same scale Perçin’s legal experts used):

+1: correct, same meaning

0: a bit awkward, but the meaning is the same

−1: wrong, the meaning changed

In [ ]:
random.seed(7)   # fixed, so the same 20 are picked every time
sample = random.sample(range(len(aug_texts)), 20)

for n, i in enumerate(sample, 1):
    orig, new = aug_sources[i].split(), aug_texts[i].split()
    print(f"#{n}   similarity {sims[i]:.3f}")
    for j, (a, b) in enumerate(zip(orig, new)):
        if a != b:
            left = " ".join(orig[max(0, j - 6):j])
            right = " ".join(orig[j + 1:j + 7])
            print(f"   ...{left} [{a} -> {b}] {right}...")
    print()

#1   similarity 0.998
   ...as trademarks , trade dress , [domain -> sphere] names , trade secrets , copyrights...
   ...patents ) ; ( 2 ) [violations -> infringement] of the api terms ; or...

#2   similarity 0.997
   ...out of , or in any [way -> means] in connection with your rights of...
   ..., or in any way in [connection -> link] with your rights of privacy and...

#3   similarity 0.993
   ...be allowed to apply for injunctive [remedies -> cure] ( or other equivalent types of...
   ...apply for injunctive remedies ( or [other -> early] equivalent types of urgent legal remedy...

#4   similarity 0.996
   ...( such as trademarks , trade [dress -> attire] , domain names , trade secrets...
   ...trademarks , trade dress , domain [names -> figure] , trade secrets , copyrights and...

#5   similarity 0.999
   ..., or controversy that arises in [connection -> link] with the agreements ( and any...

#6   similarity 0.986
   ...this policy and consent [forms -> variety] part of our websi

Changing the idea, instead of protecting words and swipping the rest we see what swips are good, approve them and ignore the rest.

**Before:** “swap anything, except the blocked words”
**Now:** “swap nothing, except the swaps we’ve approved”

In [ ]:
from collections import Counter

pairs = Counter()
for src, new in zip(aug_sources, aug_texts):
    for a, b in zip(src.split(), new.split()):
        if a != b:
            pairs[(a, b)] += 1

print("Unique swaps:", len(pairs), "\n")
for (a, b), n in pairs.most_common():
    print(f"{n:3d}   {a} -> {b}")

Unique swaps: 88 

 15   rules -> regulation
 13   connection -> link
 13   other -> early
  8   certain -> sure
  6   matter -> subject
  5   purpose -> aim
  4   otherwise -> differently
  4   types -> character
  4   also -> likewise
  3   statement -> assertion
  3   then -> so
  3   respect -> regard
  3   way -> means
  3   proceedings -> transactions
  2   means -> way
  2   forms -> variety
  2   card -> bill
  2   general -> universal
  2   remedies -> cure
  2   instance -> example
  2   informal -> intimate
  2   exception -> exclusion
  2   ever -> always
  2   violations -> infringement
  2   goods -> commodity
  2   validity -> robustness
  2   dress -> attire
  2   southern -> southerly
  2   forum -> assembly
  2   single -> individual
  2   substantive -> substantial
  2   feature -> characteristic
  2   objection -> protest
  2   policies -> insurance
  2   access -> approach
  1   reason -> cause
  1   event -> result
  1   eu -> ec
  1   part -> portion
  1   data -

**Data augmentation (tested, not used)**

I tried creating extra training examples for the three rarest types by swapping words with synonyms, following Perçin et al. (2022) while protecting legal terms.

It doubled the rare types, but a manual check showed 15 of 20 new sentences had a changed legal meaning, and only 13 of 88 word swaps were safe.

For example, “trade dress” became “trade attire” and “other than” became “early than”. The automatic similarity check still scored these broken sentences above 0.98, so it could not catch the errors.

The new sentences were not used for training. Class imbalance is handled with class weighting instead.